# 자동 인덱싱 준비 연결 — opt-in CPU 통합 시험

고정 B 저장본과 명시한 JSON3개를 실제 archive prepare API에 요청합니다. 사용자가 scan/export/import를 순서대로 호출하지 않습니다. 필요 범위만 예산 내 탐색해 자동 snapshot을 저장하고, 새 local workspace의 같은 prepare 요청으로 복원·추출합니다. 마지막으로 첫 추출본을 hash 검증해 원본 읽기 없이 재사용합니다.

세 요청은 하나의 Context와 Runtime을 공유합니다. 전체 응답 body8MiB/120HTTP/300초, source read·decode 각32MiB이며 준비 작업·snapshot·새 local DB까지 합계 disk32MiB/RSS512MiB로 제한합니다. 인증은 ledger 밖이며 최초 B 복원은 ledger 안입니다. 기존 C의 최대128MiB disk보다 작은 범위입니다. 전체 archive 완료·학습 준비·새 VM 내구성을 판정하지 않습니다.

모든 gate는 기본False입니다. 기존 B/C 출력은 입력으로만 유지하고 새 snapshot 폴더와 새 run ID를 사용합니다. 실패·중단 시 실행 셀을 다시 돌리지 않고 회수 셀만 실행합니다.


In [ ]:
# Root 검토·커밋 후 만든 C ready notebook에 pin을 채웁니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
MANIFEST_SHA256 = ""
PHASE = "C"
RUN_ID = "pose_auto_prepare_001"
EXPECTED_ANCHOR_SHA256 = ""  # 기존 B anchor를 독립 보관한 정확한 SHA
INPUT_SNAPSHOT_DIRECTORY = "/content/drive/MyDrive/reports/persistent-index/pose_persistent_86d322c_001"
OUTPUT_SNAPSHOT_DIRECTORY = "/content/drive/MyDrive/reports/persistent-index/pose_auto_prepare_001"
WORKSPACE = "/content/rewind-auto-prepare"
TARGETS = []  # ready에 검증된 B의 기존2개와 새 JSON1개의 name/sha256을 고정합니다.


## 1. 고정 코드 ZIP 업로드

첫 업로드에는 source ZIP 한 개만 선택합니다. 실행 결과 ZIP·snapshot ZIP을 올리면 거부합니다. Source SHA/revision은 ready에 고정돼 있어야 합니다.


In [ ]:
import sys, os, io, stat, json, tempfile, zipfile, shutil
from pathlib import Path, PurePosixPath
from google.colab import files

# 순차 해제 누적량과 RAM 사용량은 다릅니다. 업로드/런타임 여유를 먼저 확인합니다.
meminfo=Path("/proc/meminfo").read_text() if Path("/proc/meminfo").is_file() else ""
available_ram=next((int(row.split()[1])*1024 for row in meminfo.splitlines() if row.startswith("MemAvailable:")),None)
free_disk=shutil.disk_usage(".").free
print({"python":sys.version.split()[0],"free_disk_bytes":free_disk,"available_ram_bytes":available_ram})
if free_disk<2*1024**3 or available_ram is None or available_ram<2*1024**3:
    raise RuntimeError("가용 RAM과 임시 디스크가 각각2GiB 이상인 Colab CPU 런타임을 준비하세요.")
uploaded = files.upload()  # 여기에는 코드가 들어 있는 source ZIP 한 개만 올립니다.
if len(uploaded)!=1:
    raise ValueError("코드 source ZIP 한 개만 선택하세요.")
source_name, source_bytes = next(iter(uploaded.items()))
import hashlib
if len(SOURCE_ZIP_SHA256)!=64 or hashlib.sha256(source_bytes).hexdigest()!=SOURCE_ZIP_SHA256:
    raise ValueError("고정 source ZIP SHA256 불일치. 결과 ZIP을 올리지 마세요.")
if not source_name.lower().endswith(".zip") or len(source_bytes)>64*1024**2:
    raise ValueError("64MiB 이하의 코드 source ZIP이 필요합니다.")
excluded={".git",".venv","data","outputs","checkpoints",".aws",".codex",".agents","__pycache__"}
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    members=archive.infolist()
    if len(members)>5000 or sum(m.file_size for m in members)>128*1024**2:
        raise ValueError("코드 ZIP의 항목 수/해제 크기 상한을 초과했습니다.")
    code_entries=[m for m in members if m.filename.endswith("rewind_archive_index/__init__.py")]
    if len(code_entries)!=1:
        inner=[PurePosixPath(m.filename).name for m in members if m.filename.lower().endswith(".zip")][:6]
        if inner:
            raise ValueError("배포 묶음 ZIP입니다. 로컬에서 내부 코드 ZIP "+repr(inner)+"만 꺼내 다시 올리세요. 중첩 자동 해제는 하지 않습니다.")
        raise ValueError("rewind_archive_index/__init__.py를 포함한 코드 source ZIP이 필요합니다. Notebook/결과 ZIP과 구분하세요.")
    seen=set()
    for m in members:
        p=PurePosixPath(m.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in m.filename or any(":" in x for x in p.parts) or any(ord(c)<32 for c in m.filename) or stat.S_ISLNK(m.external_attr>>16):
            raise ValueError("안전하지 않은 코드 ZIP 경로입니다.")
        key=p.as_posix().casefold().rstrip("/")
        if key in seen:raise ValueError("중복 코드 ZIP 경로입니다.")
        seen.add(key)
    extract_root=Path(tempfile.mkdtemp(prefix="rewind_source_",dir="/content"))
    for m in members:
        parts=PurePosixPath(m.filename).parts
        if not any(p in excluded or p==".env" or p.startswith(".env.") for p in parts):
            archive.extract(m,extract_root)
roots=[p.parent.parent for p in extract_root.rglob("rewind_archive/__init__.py")]
if len(roots)!=1:raise ValueError("단일 프로젝트 source ZIP이 필요합니다.")
PROJECT_ROOT=roots[0]
if not (PROJECT_ROOT/"rewind_archive/workflow.py").is_file():
    raise ValueError("이 notebook과 맞는 최신 source ZIP이 필요합니다(workflow.py 누락).")
if any(name=="rewind_archive" or name.startswith("rewind_archive.") or name.startswith("rewind_archive_index") for name in sys.modules):
    raise RuntimeError("이전 모듈이 이미 import되어 있습니다. 기존 결과를 먼저 보관하고 런타임을 재시작한 뒤 새 source ZIP으로 시작하세요.")
os.chdir(PROJECT_ROOT)
sys.path.insert(0,str(PROJECT_ROOT))
from rewind_archive.workflow import NOTEBOOK_API_VERSION
if NOTEBOOK_API_VERSION!=10:raise RuntimeError("Notebook/source API 버전이 다릅니다.")
stamp=PROJECT_ROOT/"SOURCE_REVISION.txt"
print({"source_root":str(PROJECT_ROOT),"source_revision":stamp.read_text().strip() if stamp.exists() else "unversioned; code hashes recorded","notebook_api":NOTEBOOK_API_VERSION})
VERIFIED_SOURCE_ZIP_SHA256=hashlib.sha256(source_bytes).hexdigest()
VERIFIED_SOURCE_REVISION=stamp.read_text().strip() if stamp.is_file() else ""
del uploaded, source_bytes
if not stamp.is_file() or not SOURCE_REVISION or stamp.read_text().strip()!=SOURCE_REVISION:
    raise ValueError("고정 source revision 불일치")
from rewind_archive_index.preparation_pilot import run_preparation_integration, INTEGRATION_LIMITS
print({"persistent_drive_entrypoint":True,"fixed_phase_limits":vars(INTEGRATION_LIMITS)})


## 2. 선택 의존성

CPU 런타임에서 명시적으로 한 번 설치합니다. A/B 모두 같은 indexed_gzip1.10.3을 사용합니다. 모델·torch 설치는 필요 없습니다.


In [ ]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.run([sys.executable,"-m","pip","install","indexed_gzip==1.10.3","psutil>=5.9,<8"],check=True)


## 3. 고정 metadata와 신뢰 anchor

metadata에는 모든 원본 조각의 ID/크기/version/수정시각이 있어야 하며 source ZIP과는 별도 파일입니다. 개인 ID는 notebook에 하드코딩하지 않습니다. C에서는 기존 B에서 내려받아 별도로 보관한 anchor 파일의 SHA를 먼저 대조합니다. Drive에 같이 놓인 파일만으로 최신/분기 신뢰를 결정하지 않습니다.


In [ ]:
import hashlib, json
from rewind_archive_index.drive import make_drive_source
manifest_upload=files.upload()
if len(manifest_upload)!=1:raise ValueError("고정 archive_manifest.json 한 개를 선택하세요.")
manifest_bytes=next(iter(manifest_upload.values()))
if len(manifest_bytes)>128*1024 or len(MANIFEST_SHA256)!=64 or hashlib.sha256(manifest_bytes).hexdigest()!=MANIFEST_SHA256:
    raise ValueError("manifest 크기/SHA 불일치")
MANIFEST=json.loads(manifest_bytes)
make_drive_source(MANIFEST)
ANCHOR=None
if PHASE=="C":
    anchor_upload=files.upload()
    if len(anchor_upload)!=1:raise ValueError("고정 B anchor JSON 한 개를 선택하세요.")
    anchor_bytes=next(iter(anchor_upload.values()))
    if len(anchor_bytes)>65536 or len(EXPECTED_ANCHOR_SHA256)!=64 or hashlib.sha256(anchor_bytes).hexdigest()!=EXPECTED_ANCHOR_SHA256:
        raise ValueError("신뢰 anchor SHA 불일치")
    ANCHOR=json.loads(anchor_bytes)
print({"phase":PHASE,"run_id":RUN_ID,"metadata_checked_offline":True,"anchor_pinned":ANCHOR is not None})


## 4. Drive mount

Live SQLite는 `/content`의 로컬 workspace에서만 실행합니다. 입력 B폴더는 읽기만하고 새 C폴더에 닫힌 ZIP과 receipt만 저장합니다. 폴더의 최신 파일명을 자동으로 고르지 않습니다.


In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")


## 5. 통합 시험 한 번 실행

실제 mount와 고정 pin을 확인한 뒤 실행합니다. 최초 marker부터 provenance를 저장하고 인증 전후·자동 준비·새 workspace 복원·재사용·cleanup 단계가 표시됩니다. 자동 탐색이 예산/EOF/no-progress에 도달하면 partial로 보존합니다. 같은 실행 ID의 marker가 있으면 재실행하지 않습니다.


In [ ]:
EXECUTE_PHASE = False
def create_drive_session():
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession, Request
    from requests.adapters import HTTPAdapter
    import urllib3
    if not callable(getattr(urllib3.response.HTTPResponse,"read1",None)):
        raise RuntimeError("urllib3 raw.read1 API가 필요합니다.")
    auth.authenticate_user()
    credentials,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
    # opt-in에서는 인증용 HTTP session도 숨은 retry0으로 고정한다.
    auth_http = None
    if True:
        import requests
        auth_http=requests.Session()
        auth_http.trust_env=False
        auth_http.mount("https://",HTTPAdapter(max_retries=0))
    try:
        auth_request=Request(session=auth_http) if auth_http is not None else Request()
        credentials.refresh(auth_request)
        session=AuthorizedSession(credentials,max_refresh_attempts=0,auth_request=auth_request) if auth_http is not None else AuthorizedSession(credentials,max_refresh_attempts=0)
        if auth_http is not None:
            # custom auth_request session은 AuthorizedSession.close가 소유하지 않는다.
            original_close=session.close
            def close_sessions():
                try:original_close()
                finally:auth_http.close()
            session.close=close_sessions
        session.trust_env=False
        session.mount("https://",HTTPAdapter(max_retries=0))
    except BaseException:
        if auth_http is not None:auth_http.close()
        raise
    return session

if EXECUTE_PHASE:
    mount=Path("/content/drive")
    if not os.path.ismount(mount) or any(not Path(p).resolve().is_relative_to(mount.resolve()) for p in (INPUT_SNAPSHOT_DIRECTORY,OUTPUT_SNAPSHOT_DIRECTORY)):
        raise RuntimeError("Drive의 실제 mount와 하위 snapshot 경로를 확인하세요. 인증/marker를 시작하지 않았습니다.")
    if VERIFIED_SOURCE_ZIP_SHA256!=SOURCE_ZIP_SHA256 or VERIFIED_SOURCE_REVISION!=SOURCE_REVISION:
        raise ValueError("source pin이 변경됐습니다.")
    if hashlib.sha256(manifest_bytes).hexdigest()!=MANIFEST_SHA256 or hashlib.sha256(anchor_bytes).hexdigest()!=EXPECTED_ANCHOR_SHA256:
        raise ValueError("manifest/anchor pin이 변경됐습니다.")
    import importlib.metadata
    from rewind_archive_index.extension_records import validate_provenance
    PROVENANCE=validate_provenance({"source_revision":SOURCE_REVISION,"source_zip_sha256":SOURCE_ZIP_SHA256,
        "manifest_sha256":MANIFEST_SHA256,"anchor_sha256":EXPECTED_ANCHOR_SHA256,
        "environment":{"python":sys.version.split()[0],"indexed_gzip":importlib.metadata.version("indexed_gzip"),"psutil":importlib.metadata.version("psutil")}})
    try:
        REPORT=run_preparation_integration(run_id=RUN_ID,workspace=WORKSPACE,input_snapshot_directory=INPUT_SNAPSHOT_DIRECTORY,
            output_snapshot_directory=OUTPUT_SNAPSHOT_DIRECTORY,manifest=MANIFEST,anchor=ANCHOR,targets=TARGETS,
            provenance=PROVENANCE,session_factory=create_drive_session)
    except BaseException:
        print("중단 상태는 회수 셀로 보관하세요. 실행 셀·원본 읽기를 재시작하지 마세요.")
        raise
    print({"status":REPORT["status"],"stage":REPORT.get("stage"),"recovery":"다음 회수 셀에서 ZIP을 보관하세요."})
else:
    print("인증·원본 읽기·자동 준비를 실행하지 않았습니다.")


## 6. 독립 결과 회수

REPORT 변수 없이 디스크 marker·준비 보고서·검증된 선택 JSON만 ZIP으로 회수합니다. 인증·mount·원본 archive·DB 읽기를 재시작하지 않습니다. 실패·partial·중단도 회수하며 손상/누락 marker는 진단으로 표시합니다. 고정 run/pin과 다른 자료는 거부합니다. 같은 ZIP은 검증 후 다시 내려받을 수 있습니다.


In [ ]:
DOWNLOAD_REPORT = False
if DOWNLOAD_REPORT:
    # 실행 셀/REPORT/인증 세션에 의존하지 않는 독립 회수 진입점입니다.
    import sys, hashlib, importlib.metadata
    from rewind_archive_index.extension_records import validate_provenance
    from rewind_archive_index.preparation_pilot import recover_preparation_integration
    from google.colab import files
    def recovery_version(name):
        try:return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:return "not_installed"
    recovery_pins=validate_provenance({"source_revision":SOURCE_REVISION,"source_zip_sha256":SOURCE_ZIP_SHA256,
        "manifest_sha256":MANIFEST_SHA256,"anchor_sha256":EXPECTED_ANCHOR_SHA256,
        "environment":{"python":sys.version.split()[0],"indexed_gzip":recovery_version("indexed_gzip"),"psutil":recovery_version("psutil")}})
    RECOVERY=recover_preparation_integration(workspace=WORKSPACE,run_id=RUN_ID,provenance=recovery_pins)
    from pathlib import Path
    print({**RECOVERY,"sha256":hashlib.sha256(Path(RECOVERY["path"]).read_bytes()).hexdigest()})
    files.download(RECOVERY["path"])
    REPORT_BUNDLE_REQUESTED=True


## 7. 성공 자료 보관 뒤 선택 flush

회수 ZIP을 보관한 뒤 이 별도 셀의 gate만 켭니다. 성공한 marker와 검증된 ZIP이 필요합니다. 실패·중단 상태는 런타임과 자료를 유지합니다. C 완료 뒤 추가 VM 교체 시험은 요구하지 않습니다.


In [ ]:
FLUSH_AND_UNMOUNT = False
if FLUSH_AND_UNMOUNT:
    import json
    from pathlib import Path
    if not globals().get("REPORT_BUNDLE_REQUESTED") or not globals().get("RECOVERY"):
        raise RuntimeError("회수 셀에서 성공 ZIP을 먼저 내려받아 보관하세요.")
    verified=recover_preparation_integration(workspace=WORKSPACE,run_id=RUN_ID,provenance=recovery_pins)
    import zipfile
    with zipfile.ZipFile(verified["path"]) as archive:
        status=json.loads(archive.read("report.json"))["status"]
    if status!="passed":raise RuntimeError("실패·중단 런타임을 보존하세요.")
    from google.colab import drive
    try:
        drive.flush_and_unmount(timeout_ms=60000)
        FLUSH_REPORT={"flush":"returned","timeout_ms":60000,"persistence_proven":False}
    except Exception:
        FLUSH_REPORT={"flush":"failed","timeout_ms":60000,"persistence_proven":False,"action":"현재 런타임과 보고서를 보존하고 테스트 세션에 전달"}
    FLUSH_PATH=Path(WORKSPACE)/(RUN_ID+"-"+PHASE+"-flush.json")
    with FLUSH_PATH.open("x",encoding="utf-8") as stream:json.dump(FLUSH_REPORT,stream,ensure_ascii=False)
    print(FLUSH_REPORT);files.download(str(FLUSH_PATH))
